## Import Libraries
We import the necessary Python modules and PyTorch components that will be used throughout our project.  

**random** :   
    - Useful for controlling random behavior, though not heavily used here.  

**NumPy** :   
    - Complementary numerical library for array operations.  

**PyTorch (torch)** :   
    - Core library for handling tensors, automatic differentiation, and GPU computation. 


**nn and optim** :   
    - Provide pre-built neural network layers (e.g., Conv2d, Linear) and optimization methods (e.g., Adam).  


**torchvision.datasets** :   
    - Simplifies loading common datasets (such as FashionMNIST).  


**torchvision.transforms** :   
    - Preprocessing utilities (e.g., ToTensor()).  


**DataLoader** :   
    - Groups samples into batches for efficient training.  


**torchmetrics** :   
    - Offers ready-to-use metrics (Accuracy, Precision, Recall) that integrate well with PyTorch.  

In [20]:
# -------------------------
# Standard Library Imports
# -------------------------
import random  # Python's built-in random module

# -------------------------
# Third-Party Library Imports
# -------------------------
import numpy as np            # NumPy for numerical computations
import torch                  # PyTorch for tensor operations
import torch.nn as nn         # Neural network layers and functions
import torch.optim as optim   # Optimization algorithms

# -------------------------
# TorchVision and Transforms
# -------------------------
from torchvision import datasets              # Popular datasets (e.g., ImageFolder)
from torchvision.transforms import transforms # Image transformations (e.g., Resize, Normalize)

# -------------------------
# PyTorch Data Utilities
# -------------------------
from torch.utils.data import Dataset, DataLoader  # Custom dataset creation and batch loading

# -------------------------
# TorchMetrics for Evaluation
# -------------------------
from torchmetrics import Accuracy, Precision, Recall  # Standard evaluation metrics for classification

## Load the Dataset
In this step, we download and load the **`FashionMNIST`** dataset, split into a training set (train=True) and a test set (train=False). We specify a root directory (./data) where the data will be stored, and use **`download=True`** to automatically fetch the dataset if it does not exist locally. By applying **`transforms.ToTensor()`**, each image is converted from a PIL image to a normalized PyTorch tensor, simplifying the process of feeding image data into our model.

In [21]:
# -------------------------
# Load the Training and Test Datasets
# -------------------------
# FashionMNIST is automatically downloaded to './data' (if not already present).
# The transform 'ToTensor()' converts PIL images to PyTorch Tensors.
train_data = datasets.FashionMNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transforms.ToTensor()
)
test_data = datasets.FashionMNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transforms.ToTensor()
)

## Inspect the Dataset
Here, we gather crucial information about our datasets before training. We confirm the number of images available in each split and quickly inspect the shapes of the first training and test samples to ensure they match the expected **`one-channel`**, **`28×28`** **`format`**. We calculate the **`total number of pixels (28 * 28 = 784)`** to understand the raw input dimension for fully connected layers, and we also print out the class labels recognized by **`FashionMNIST`**, ultimately confirming there are **`10 unique categories`** to classify.

In [22]:
# -------------------------
# Basic Dataset Information
# -------------------------
# Print the number of samples in each dataset
print(f"Number of samples in train_data: {len(train_data)}")
print(f"Number of samples in test_data: {len(test_data)}", "\n")

# Inspect the first sample from each dataset
image, label = train_data[0]
image2, label2 = test_data[0]

# Print the shapes of the images
print(f"Train image shape: {image.shape}")
print(f"Test image shape: {image2.shape}", "\n")

# Calculate and print the number of features per image
# For FashionMNIST, each image is 1 channel with size 28 x 28.
num_features = image.shape[1] * image.shape[2]
print(f"Number of features per image: {num_features}", "\n")

# Check and print the available class labels
print(f"Unique class labels in train: {train_data.classes}")
print(f"Unique class labels in test: {test_data.classes}", "\n")
classes = train_data.classes

num_classes = len(train_data.classes)
print(f"Number of unique class labels in train: {num_classes}")

Number of samples in train_data: 60000
Number of samples in test_data: 10000 

Train image shape: torch.Size([1, 28, 28])
Test image shape: torch.Size([1, 28, 28]) 

Number of features per image: 784 

Unique class labels in train: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
Unique class labels in test: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'] 

Number of unique class labels in train: 10


## Defining the Convolutional Neural Network (CNN) 

This step constructs a **`convolutional neural network (CNN)`** for image classification. It consists of **`two convolutional blocks`** (each followed by ReLU activation and max pooling) that progressively extract spatial features from the input. Using **`Flatten`** transforms the 2D feature maps into a 1D vector, suitable for the final linear (fully connected) layer. We dynamically determine the output size of the feature extraction part by passing a dummy tensor, ensuring that our linear layer’s input dimension matches the extracted feature dimension.


In [23]:
# -------------------------
# Define the CNN Class
# -------------------------
class CNN_net(nn.Module):
    def __init__(self, num_classes):
        super(CNN_net, self).__init__()

        # Feature extraction layers: 
        # 1) Convolution layer with 16 output channels, kernel 3x3, stride=1, padding=1
        # 2) ReLU activation function
        # 3) 2x2 Max Pooling with stride=2
        # 4) Another Conv layer (16 -> 32 channels), ReLU, and 2x2 MaxPool
        # 5) Flatten the 2D feature maps to a 1D vector
        self.feature_extraction = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Flatten()  # Flatten feature maps into a single vector
        )

        # Dynamically compute the number of features after the feature extraction layers.
        with torch.no_grad():
            image_size = train_data[0][0].shape[1]  # For FashionMNIST, this should be 28.
            sample_input = torch.zeros(1, 1, image_size, image_size)  # A dummy input
            feature_map = self.feature_extraction(sample_input)
            num_features = feature_map.shape[1]  # Flattened feature size
            # print("Feature map shape:", feature_map.shape)  # Debugging output

        # A single fully connected layer to map extracted features to class scores.
        self.classifier = nn.Linear(num_features, num_classes)

    def forward(self, x):
        """
        Forward pass:
        1) Extract features through the convolution, activation, and pooling layers.
        2) Flatten the features into a 1D vector.
        3) Pass the flattened vector to the fully connected layer for classification.
        """
        x = self.feature_extraction(x)  # Convolution & Pooling
        x = self.classifier(x)          # Linear layer
        return x


## Prepare the Training DataLoader

We now wrap our training dataset in a **`DataLoader`**, which handles batching and data shuffling. Each iteration will yield a batch of **`10`** samples (images and labels), and **`shuffle=True`** ensures that data is randomized each epoch, improving the robustness of training by reducing any unintended ordering effects in the dataset.

In [24]:
# -------------------------
# Define the Training DataLoader
# -------------------------
# Create batches of size 10 from the training dataset, and shuffle the data.
dataloader_train = DataLoader(
    train_data,
    batch_size=10,
    shuffle=True,
)

## Define the Training Function
In this function, we iterate over the **`training data`** for a specified number of epochs, computing and **`backpropagating`** the **`cross-entropy loss`** after each forward pass. The criterion is **`nn.CrossEntropyLoss`**, well-suited to multi-class classification. Each batch updates the model’s parameters through the optimizer (**`optimizer.step()`**), and we track the average loss across all batches per epoch, providing a measure of training progress.

In [25]:
# -------------------------
# Define the Training Function
# -------------------------
def train_model(optimizer, net, num_epochs):
    """
    Trains a given neural network for a specified number of epochs.
    
    Parameters:
    -----------
    optimizer: Torch Optimizer
        The optimizer responsible for updating model parameters.
    net: nn.Module
        The neural network model to be trained.
    num_epochs: int
        Number of training epochs.
    
    Returns:
    --------
    None
    """
    # Initialize the loss function
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        running_loss = 0.0  # Accumulate loss over each epoch
        num_processed = 0   # Track how many samples have been processed

        for features, labels in dataloader_train:
            # Zero out the gradients from the previous iteration
            optimizer.zero_grad()
            # Forward pass
            output = net(features)
            # Calculate loss
            loss = criterion(output, labels)
            # Backward pass
            loss.backward()
            # Update weights
            optimizer.step()
            # Accumulate the loss
            running_loss += loss.item()
            # Count the number of processed samples
            num_processed += len(labels)

        # Print the average training loss for this epoch
        train_loss = running_loss / len(dataloader_train)
        print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.4f}")

## Instantiate and Train the Model

Here, we create our **`CNN instance`** and an **`Adam optimizer`**, which is known for its adaptive learning rate and efficient convergence properties. We set the **`learning rate to 0.01`** and call our training function for a single epoch (though more epochs are typically needed for better accuracy). During training, the optimizer updates the CNN’s weights, reducing the cross-entropy loss and hopefully improving model performance.



In [26]:
# -------------------------
# Instantiate the Model and Optimizer
# -------------------------
net = CNN_net(num_classes)             # Create an instance of the CNN
optimizer = optim.Adam(net.parameters(), lr=0.01)  # Use Adam optimizer with a learning rate of 0.01

# -------------------------
# Train the Model
# -------------------------
train_model(
    optimizer=optimizer,
    net=net,
    num_epochs=10
)

Epoch 1/10, Training Loss: 0.4837
Epoch 2/10, Training Loss: 0.4258
Epoch 3/10, Training Loss: 0.4172
Epoch 4/10, Training Loss: 0.4124
Epoch 5/10, Training Loss: 0.4046
Epoch 6/10, Training Loss: 0.4044
Epoch 7/10, Training Loss: 0.4022
Epoch 8/10, Training Loss: 0.3974
Epoch 9/10, Training Loss: 0.3952
Epoch 10/10, Training Loss: 0.3966


## Define Evaluation Metrics

Next, we set up three metrics—**`accuracy`**, **`precision`**, and **`recall`**—to assess different aspects of our model’s predictions. Specifying **`task='multiclass'`** and the number of classes ensures the metrics handle multi-class classification correctly. We use **`average=None`** for both precision and recall to capture per-class performance, allowing us to see if the model struggles with any particular category of clothing.

In [27]:
# -------------------------
# Define the Evaluation Metrics
# -------------------------
# Using TorchMetrics to calculate Accuracy, Precision, and Recall.
accuracy_metric = Accuracy(task='multiclass', num_classes=num_classes)
precision_metric = Precision(task='multiclass', num_classes=num_classes, average=None)
recall_metric = Recall(task='multiclass', num_classes=num_classes, average=None)

## Evaluate on the Test Set
We switch the model to evaluation mode using net.eval(), which disables training behaviors like dropout. We also create a test **`DataLoader`** and run inference in a **`torch.no_grad()`** context to avoid computing gradients. For each batch, we **`reshape`** the images if needed and make predictions by choosing the class with the highest logit (**`argmax`**). The predictions and labels are then passed to our previously defined TorchMetrics objects to accumulate performance statistics across the entire test dataset.

In [28]:
# -------------------------
# Evaluate on Test Set
# -------------------------
net.eval()  # Set the network to evaluation mode (disable dropout, batch norm, etc.)
predictions = []

# Create a DataLoader for the test dataset (not defined above, so define it here)
dataloader_test = DataLoader(
    test_data,
    batch_size=10,
    shuffle=False,
)

# Disable gradient tracking for evaluation
with torch.no_grad():
    image_size = train_data[0][0].shape[1]  # This should be 28 for FashionMNIST
    for i, (features, labels) in enumerate(dataloader_test):
        # Forward pass; ensure features are in the correct shape (N, C, H, W)
        output = net.forward(features.reshape(-1, 1, image_size, image_size))
        # Get the predicted class by picking the highest logit
        cat = torch.argmax(output, dim=-1)
        # Store predictions
        predictions.extend(cat.tolist())
        # Update evaluation metrics
        accuracy_metric(cat, labels)
        precision_metric(cat, labels)
        recall_metric(cat, labels)


## Compute and Print Final Metrics

Finally, we call the **`.compute()`** method on each metric to convert the accumulated statistics into concrete values. We print the overall **`accuracy`**, followed by **`per-class precision`** and **`recall`**, allowing us to see which classes the model predicts well and where it might be making mistakes. This completes our pipeline for training and evaluating the **`CNN`** on **`FashionMNIST`**.

In [29]:
# -------------------------
# Compute the Final Metrics
# -------------------------
accuracy = accuracy_metric.compute().item()
precision = precision_metric.compute().tolist()
recall = recall_metric.compute().tolist()

# -------------------------
# Print Evaluation Results
# -------------------------
print('Accuracy:', accuracy, "\n")
print('Precision (per class):')
for class_name, prec in zip(classes, precision):
    print(f'{class_name} : {prec:.4f}')

print("\n", 'Recall (per class):')
for class_name, rec in zip(classes, recall):
    print(f'{class_name} : {rec:.4f}')

Accuracy: 0.847599983215332 

Precision (per class):
T-shirt/top : 0.8129
Trouser : 0.9798
Pullover : 0.6463
Dress : 0.7970
Coat : 0.7683
Sandal : 0.9326
Shirt : 0.7008
Sneaker : 0.9315
Bag : 0.9624
Ankle boot : 0.9671

 Recall (per class):
T-shirt/top : 0.8080
Trouser : 0.9680
Pullover : 0.8660
Dress : 0.9150
Coat : 0.6200
Sandal : 0.9680
Shirt : 0.5060
Sneaker : 0.9390
Bag : 0.9460
Ankle boot : 0.9400
